# AIMO3 Competition Notebook — Nemotron 3 Nano 30B A3B BF16 + Python Tooling

This notebook is built for **Kaggle notebook submission** to **AI Mathematical Olympiad – Progress Prize 3** using
your uploaded **Nemotron BF16 shards** from personal Kaggle datasets.

## What this notebook changes

- uses your **exact Kaggle personal dataset paths** for the Nemotron model
- assembles the model under `/kaggle/working/nemotron-model` via symlinks
- supports the **official Nemotron vLLM tool-calling / reasoning parser setup**
- uses **mixed attempt policies**:
  - high-entropy reasoning attempts (`temperature=1.0`, `top_p=1.0`)
  - tool-enabled attempts (`temperature=0.6`, `top_p=0.95`)
- keeps **persistent Jupyter kernels** for fast tool use
- uses **strict answer extraction**
- uses **entropy-aware weighted voting** when logprobs are available
- avoids hard-coding single-GPU execution
- keeps a strong AIMO-specific system prompt focused on exact reasoning and answer extraction

## Important assumption

This version is wired to these Kaggle dataset parts:

- `/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b1`
- `/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b2`
- `/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b3`
- `/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b4`
- `/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b5`

If those dataset names change, update the model path cell before submission.


In [ ]:
%pip uninstall --yes keras matplotlib scikit-learn tensorflow

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:


In [ ]:
import os

# Must be set BEFORE importing vllm / starting the server.
# This keeps the FlashInfer FP8 MoE path disabled so the notebook stays on BF16.
os.environ["VLLM_USE_FLASHINFER_MOE_FP8"] = "0"
os.environ.pop("VLLM_FLASHINFER_MOE_BACKEND", None)

# Keep logs visible and deterministic.
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"


In [ ]:
import os
import sys
import tarfile
import subprocess
from pathlib import Path
from importlib import metadata as importlib_metadata
REQUIRED_VLLM = '0.19.0'
SETUP_DIR = Path('/kaggle/working/aimo_setup')
UTILS_DIR = SETUP_DIR / 'utils_wheels'
VLLM_WHEELS_PREFERRED = ['/kaggle/input/datasets/samiulislam180041221/vllm-latest-wheels', '/kaggle/input/vllm-latest-wheels']
UTILS_ARCHIVE_PREFERRED = ['/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz', '/kaggle/input/aimo-3-utils/wheels.tar.gz', '/kaggle/input/aimo3-utils/wheels.tar.gz', '/kaggle/input/aimo-utils/wheels.tar.gz', '/kaggle/input/utils/wheels.tar.gz']

def _version_tuple(version_str: str) -> tuple[int, ...]:
    parts = []
    for token in version_str.split('.'):
        token = ''.join((ch for ch in token if ch.isdigit()))
        parts.append(int(token) if token else 0)
    return tuple(parts)

def _needs_vllm_install(min_version: str=REQUIRED_VLLM) -> bool:
    try:
        current = importlib_metadata.version('vllm')
        return _version_tuple(current) < _version_tuple(min_version)
    except importlib_metadata.PackageNotFoundError:
        return True

def _find_vllm_wheels() -> Path | None:
    for path in VLLM_WHEELS_PREFERRED:
        p = Path(path)
        if p.is_dir() and any(p.glob('vllm*.whl')):
            return p
    root = Path('/kaggle/input')
    if root.exists():
        for candidate in root.rglob('vllm*.whl'):
            return candidate.parent
    return None

def _find_utils_archive() -> Path | None:
    for path in UTILS_ARCHIVE_PREFERRED:
        p = Path(path)
        if p.exists():
            return p
    root = Path('/kaggle/input')
    if root.exists():
        for candidate in root.rglob('wheels.tar.gz'):
            return candidate
        for candidate in root.rglob('*wheels*.tar.gz'):
            return candidate
    return None
SETUP_DIR.mkdir(parents=True, exist_ok=True)
vllm_dir = _find_vllm_wheels()
if vllm_dir is None and _needs_vllm_install():
    raise RuntimeError(f"vLLM >= {REQUIRED_VLLM} is required but no wheels found. Attach 'samiulislam180041221/vllm-latest-wheels' as an input dataset.")
utils_archive = _find_utils_archive()
if utils_archive is not None and (not UTILS_DIR.exists()):
    with tarfile.open(utils_archive, 'r:gz') as tar:
        tar.extractall(UTILS_DIR)
if _needs_vllm_install():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(vllm_dir), '--no-deps', 'vllm==0.19.0'], check=True)
if vllm_dir is not None:
    excluded_prefixes = ('protobuf-', 'opentelemetry_api-', 'opentelemetry_sdk-', 'opentelemetry_proto-', 'opentelemetry_exporter_otlp-', 'opentelemetry_exporter_otlp_proto_common-', 'opentelemetry_exporter_otlp_proto_grpc-', 'opentelemetry_exporter_otlp_proto_http-', 'opentelemetry_semantic_conventions-', 'opentelemetry_semantic_conventions_ai-')
    safe_wheels = []
    for whl in sorted(Path(vllm_dir).glob('*.whl')):
        name = whl.name.lower()
        if name.startswith('vllm-'):
            continue
        if name.startswith(excluded_prefixes):
            continue
        safe_wheels.append(str(whl))
    if safe_wheels:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', *safe_wheels], check=True)
if UTILS_DIR.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(UTILS_DIR), 'openai', 'polars', 'transformers', 'jupyter_client', 'packaging'], check=True)
enc_dir = UTILS_DIR / 'tiktoken_encodings'
if enc_dir.exists():
    os.environ['TIKTOKEN_ENCODINGS_BASE'] = str(enc_dir)
import google.protobuf
if int(google.protobuf.__version__.split('.')[0]) >= 6:
    raise RuntimeError('protobuf must stay < 6')
import cbor2
import msgspec
import vllm

In [ ]:
import os
import gc
import re
import sys
import json
import math
import time
import glob
import queue
import signal
import shutil
import threading
import subprocess
import contextlib
from pathlib import Path
from typing import Any, Optional
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
import torch
import pandas as pd
import polars as pl
from IPython.display import display
from packaging import version
from openai import OpenAI
from jupyter_client import KernelManager
from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server
import vllm
if version.parse(vllm.__version__) < version.parse('0.12.0'):
    raise RuntimeError(f'Need vLLM >= 0.12.0, found {vllm.__version__}')
GPU_COUNT = torch.cuda.device_count()

In [ ]:
PARSER_PATH = Path('/kaggle/working/nano_v3_reasoning_parser.py')
parser_code = '\nfrom vllm.reasoning.abs_reasoning_parsers import ReasoningParserManager\nfrom vllm.reasoning.deepseek_r1_reasoning_parser import DeepSeekR1ReasoningParser\n\n@ReasoningParserManager.register_module("nano_v3")\nclass NanoV3ReasoningParser(DeepSeekR1ReasoningParser):\n    def extract_reasoning(self, model_output, request):\n        reasoning_content, final_content = super().extract_reasoning(model_output, request)\n        if (\n            hasattr(request, "chat_template_kwargs")\n            and request.chat_template_kwargs\n            and request.chat_template_kwargs.get("enable_thinking") is False\n            and final_content is None\n        ):\n            reasoning_content, final_content = final_content, reasoning_content\n        return reasoning_content, final_content\n'.strip() + '\n'
PARSER_PATH.write_text(parser_code, encoding='utf-8')

In [ ]:
import os
import torch
NEMO_DIR = '/kaggle/working/nemotron-model'
os.makedirs(NEMO_DIR, exist_ok=True)
for part in ['/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b1', '/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b2', '/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b3', '/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b4', '/kaggle/input/datasets/samiulislam180041221/nemotron-nano-bf16-b5']:
    if not os.path.exists(part):
        raise FileNotFoundError(f'Missing dataset part: {part}')
    for fname in os.listdir(part):
        if fname == 'dataset-metadata.json':
            continue
        src = os.path.join(part, fname)
        dst = os.path.join(NEMO_DIR, fname)
        if not os.path.exists(dst):
            os.symlink(src, dst)
required_files = ['config.json', 'tokenizer.json']
missing = [f for f in required_files if not os.path.exists(os.path.join(NEMO_DIR, f))]
if missing:
    raise FileNotFoundError(f'Nemotron model directory is missing required files: {missing}')
os.environ['AIMO_MODEL_PATH'] = NEMO_DIR
MODEL_PATH = NEMO_DIR
MODEL_VARIANT = 'bf16'
os.environ['VLLM_USE_FLASHINFER_MOE_FP8'] = '0'
os.environ.pop('VLLM_FLASHINFER_MOE_BACKEND', None)

def detect_tp_size() -> int:
    env_value = os.getenv('AIMO_TP_SIZE')
    if env_value:
        return max(1, int(env_value))
    return max(1, torch.cuda.device_count())
TP_SIZE = detect_tp_size()

class CFG:
    model_path = MODEL_PATH
    model_variant = MODEL_VARIANT
    served_model_name = 'nemotron-nano'
    system_prompt = 'You are an elite olympiad-level mathematical reasoner solving answer-only AIMO problems. Your objective is maximum correctness. Never guess, never pattern-match loosely, and never invent a lemma, identity, or computation. Every nontrivial claim must be derived, justified, or verified.\n\nCore protocol:\n1. Restate the exact target quantity, including any modulus or answer-format requirement.\n2. Identify the mathematical domain and search for the governing structure first: invariants, factorization, symmetry, bijections, extremal arguments, recurrences, standard lemmas, or geometric relations.\n3. Keep the derivation exact. Prefer symbolic manipulation and exact modular arithmetic over decimal approximations.\n4. Use the Python tool for exact support whenever arithmetic risk, symbolic expansion, modular evaluation, prime/divisor checks, recurrence steps, or bounded finite verification would improve reliability.\n5. Never use brute force over huge spaces and never use empirical pattern spotting as the main justification.\n6. Before finalizing, verify that all problem constraints are satisfied and that any required remainder is reduced correctly.\n7. Do not finalize until you are confident that the derived value is legitimate. If uncertain, keep reasoning or verify with the tool instead of guessing.\n8. The final response must contain exactly one boxed integer in the form \\boxed{N}, with N between 0 and 99999.'
    reasoning_suffix = 'Solve in a proof-oriented way. Prefer a robust derivation over a clever guess. If one line of attack stalls, try a second principled route.'
    tool_suffix = 'You have access to a stateful Python verification tool. Use it as an exact checker, not as a substitute for reasoning. Before each tool call, state what is being checked. After each tool result, interpret it and continue the derivation. Use tools for exact symbolic algebra, modular arithmetic, factorizations, divisor/prime checks, recurrences, or bounded finite searches that validate a derived claim. For any nontrivial arithmetic or finite-case verification, prefer checking with the tool rather than trusting mental math.'
    progress_prompt = 'Continue carefully. If a nontrivial computation, modular check, or bounded case analysis remains, use the Python tool rather than guessing. When you are fully certain, end with exactly one boxed integer \\boxed{N}.'
    finalization_prompt = 'Using the complete reasoning and any prior Python tool outputs above, return exactly one final boxed integer \\boxed{N}. Do not introduce new assumptions. Do not call tools in this response. If a modulus or remainder was requested, return the exact remainder.'
    tool_prompt = 'Stateful Python notebook tool for exact math support. Available modules include: math, itertools, collections, fractions, functools, statistics, numpy, sympy, mpmath. Prefer exact arithmetic and print all results you want the model to see.'
    seed = 42
    workers = int(os.getenv('AIMO_WORKERS', '8'))
    early_stop = 999
    turns = int(os.getenv('AIMO_TURNS', '24'))
    tensor_parallel_size = TP_SIZE
    max_num_seqs = int(os.getenv('AIMO_MAX_NUM_SEQS', '8' if TP_SIZE > 1 else '8'))
    context_tokens = int(os.getenv('AIMO_CONTEXT_TOKENS', '262144'))
    max_tokens_reasoning = int(os.getenv('AIMO_MAX_TOKENS_REASONING', '131072'))
    max_tokens_tool = int(os.getenv('AIMO_MAX_TOKENS_TOOL', '131072'))
    context_buffer_tokens = int(os.getenv('AIMO_CONTEXT_BUFFER_TOKENS', '2048'))
    min_generation_tokens = int(os.getenv('AIMO_MIN_GENERATION_TOKENS', '128'))
    tool_turn_buffer_tokens = int(os.getenv('AIMO_TOOL_TURN_BUFFER_TOKENS', '1536'))
    chat_turn_buffer_tokens = int(os.getenv('AIMO_CHAT_TURN_BUFFER_TOKENS', '768'))
    gpu_memory_utilization = float(os.getenv('AIMO_GPU_MEM_UTIL', '0.90'))
    finalization_max_tokens = int(os.getenv('AIMO_FINALIZATION_MAX_TOKENS', '256'))
    high_diversity_min_p = float(os.getenv('AIMO_HIGH_DIVERSITY_MIN_P', '0.02'))
    logprobs = False
    top_logprobs = 5
    dtype = 'bfloat16'
    kv_cache_dtype = 'auto'
    stream_interval = 200
    enable_prefix_caching = os.getenv('AIMO_ENABLE_PREFIX_CACHING', '0') == '1'
    enable_async_scheduling = os.getenv('AIMO_ENABLE_ASYNC_SCHEDULING', '1') == '1'
    fallback_max_num_seqs = int(os.getenv('AIMO_FALLBACK_MAX_NUM_SEQS', '4'))
    server_timeout = 1200
    session_timeout = 1200
    sandbox_timeout = 60
    jupyter_timeout = 120
    notebook_limit = int(os.getenv('AIMO_NOTEBOOK_LIMIT_SEC', str(5 * 60 * 60 - 10 * 60)))
    high_problem_timeout = 780
    base_problem_timeout = 180
    min_answer = 0
    max_answer = 999
    attempt_plans = [{'name': f'T{i + 1:02d}', 'use_tools': True, 'temperature': 0.6, 'top_p': 0.95, 'min_p': None, 'max_tokens': max_tokens_tool, 'prompt_suffix': tool_suffix} for i in range(32)]
    attempts = len(attempt_plans)

In [ ]:
class AIMO3Sandbox:
    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float):
        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None

        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
        env["PYDEVD_WARN_EVALUATION_TIMEOUT"] = "0"
        env["JUPYTER_PLATFORM_DIRS"] = "1"
        env["PYTHONWARNINGS"] = "ignore"
        env["MPLBACKEND"] = "Agg"

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]
        self._km.start_kernel(env=env, extra_arguments=["--Application.log_level=CRITICAL"])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(self._bootstrap_code())

    @staticmethod
    def _bootstrap_code() -> str:
        return (
            "import math\n"
            "import numpy as np\n"
            "import sympy\n"
            "import itertools\n"
            "import collections\n"
            "import fractions\n"
            "import functools\n"
            "import statistics\n"
            "import mpmath\n"
            "mpmath.mp.dps = 80\n"
        )

    def _format_error(self, traceback: list[str]) -> str:
        clean_lines = []
        for frame in traceback:
            clean_frame = re.sub(r"\x1b\[[0-9;]*m", "", frame)
            if 'File "' in clean_frame and "ipython-input" not in clean_frame:
                continue
            clean_lines.append(clean_frame)
        return "".join(clean_lines)

    def execute(self, code: str, timeout: Optional[float] = None) -> str:
        client = self._client
        effective_timeout = timeout or self._default_timeout

        msg_id = client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout_parts = []
        stderr_parts = []
        start_time = time.time()

        while True:
            if time.time() - start_time > effective_timeout:
                self._km.interrupt_kernel()
                return f"[ERROR] Execution timed out after {effective_timeout} seconds"

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get("parent_header", {}).get("msg_id") != msg_id:
                continue

            msg_type = msg.get("msg_type")
            content = msg.get("content", {})

            if msg_type == "stream":
                text = content.get("text", "")
                if content.get("name") == "stdout":
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)

            elif msg_type == "error":
                traceback_list = content.get("traceback", [])
                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {"execute_result", "display_data"}:
                data = content.get("data", {})
                text = data.get("text/plain")
                if text:
                    stdout_parts.append(text if text.endswith("\n") else f"{text}\n")

            elif msg_type == "status" and content.get("execution_state") == "idle":
                break

        stdout = "".join(stdout_parts)
        stderr = "".join(stderr_parts)
        if stderr:
            return f"{stdout.rstrip()}\n{stderr}" if stdout else stderr
        return stdout if stdout.strip() else "[WARN] No output. Use print() to see results."

    def reset(self) -> None:
        self.execute("%reset -f\n" + self._bootstrap_code())

    def close(self) -> None:
        # Kaggle/Papermill teardown is sensitive to async kernel shutdown.
        # Do not explicitly shutdown the kernel here.
        try:
            if self._client is not None:
                with contextlib.suppress(Exception):
                    self._client.stop_channels()
        finally:
            self._client = None
    
        # Important: do NOT call shutdown_kernel() or cleanup_resources()
        # Let Kaggle destroy the container/process tree naturally.
        self._km = None

    def __del__(self):
        with contextlib.suppress(Exception):
            self.close()


class AIMO3Tool:
    def __init__(self, local_jupyter_timeout: float, sandbox: AIMO3Sandbox | None = None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._jupyter_session = sandbox
        self._owns_session = sandbox is None
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:
        lines = code.strip().split("\n")
        if not lines:
            return code
        last_line = lines[-1].strip()
        if not last_line:
            return code
        if last_line.startswith("#"):
            return code
        if "print(" in last_line or last_line.startswith(("import ", "from ")):
            return code
        lines[-1] = f"print({last_line})"
        return "\n".join(lines)

    @property
    def supported_tool_names(self) -> tuple[str, ...]:
        return ("stateful_python_code_exec")

    @property
    def tools(self) -> list[dict]:
        return [
            {
                "type": "function",
                "function": {
                    "name": "stateful_python_code_exec",
                    "description": CFG.tool_prompt,
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "code": {
                                "type": "string",
                                "description": "Python code to execute in the stateful notebook environment."
                            }
                        },
                        "required": ["code"],
                        "additionalProperties": False,
                    },
                },
            }
        ]

    def run(self, code: str) -> str:
        self._ensure_session()
        final_script = self._ensure_last_print(code)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script, timeout=self._local_jupyter_timeout)
            except TimeoutError as exc:
                output = f"[ERROR] {exc}"
        return output


In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int=8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://127.0.0.1:{port}/v1'
        self.api_key = 'sk-local'
        self.server_process = None
        self.log_file = None
        self.client = None
        set_seed(self.cfg.seed)
        self._preload_model_weights()
        self._launch_server_with_retries()
        self._initialize_kernels()
        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _preload_model_weights(self) -> None:
        start_time = time.time()
        files_to_load = []
        total_size = 0
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
        with ThreadPoolExecutor(max_workers=min(self.cfg.workers, 8)) as executor:
            list(executor.map(_read_file, files_to_load))
        elapsed = time.time() - start_time

    def _tail_logs(self, max_chars: int=20000) -> str:
        log_path = Path('/kaggle/working/vllm_server.log')
        if not log_path.exists():
            return '[No vLLM log file found.]'
        text = log_path.read_text(encoding='utf-8', errors='ignore')
        return text[-max_chars:] if len(text) > max_chars else text

    def _cleanup_server(self) -> None:
        if getattr(self, 'server_process', None) is not None:
            with contextlib.suppress(Exception):
                os.killpg(os.getpgid(self.server_process.pid), signal.SIGTERM)
            with contextlib.suppress(Exception):
                self.server_process.wait(timeout=30)
            self.server_process = None
        if getattr(self, 'log_file', None) is not None:
            with contextlib.suppress(Exception):
                self.log_file.flush()
            with contextlib.suppress(Exception):
                self.log_file.close()
            self.log_file = None

    def _server_env(self, safer: bool=False) -> dict:
        env = os.environ.copy()
        env['VLLM_USE_FLASHINFER_MOE_FP8'] = '0'
        env.pop('VLLM_FLASHINFER_MOE_BACKEND', None)
        env.setdefault('VLLM_LOGGING_LEVEL', 'INFO')
        return env

    def _build_server_cmd(self, safer: bool=False) -> list[str]:
        max_num_seqs = self.cfg.max_num_seqs if not safer else min(self.cfg.max_num_seqs, self.cfg.fallback_max_num_seqs)
        cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--seed', str(self.cfg.seed), '--model', self.cfg.model_path, '--served-model-name', self.cfg.served_model_name, '--tensor-parallel-size', str(self.cfg.tensor_parallel_size), '--max-num-seqs', str(max_num_seqs), '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization), '--host', '0.0.0.0', '--port', str(self.port), '--dtype', self.cfg.dtype, '--kv-cache-dtype', self.cfg.kv_cache_dtype, '--mamba-ssm-cache-dtype', 'float32', '--max-model-len', str(self.cfg.context_tokens), '--stream-interval', str(self.cfg.stream_interval), '--disable-log-stats', '--trust-remote-code', '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_coder', '--reasoning-parser-plugin', str(PARSER_PATH), '--reasoning-parser', 'nano_v3']
        if self.cfg.enable_async_scheduling and (not safer):
            cmd.append('--async-scheduling')
        if self.cfg.enable_prefix_caching and (not safer):
            cmd.append('--enable-prefix-caching')
        return cmd

    def _start_server(self, safer: bool=False, attempt_name: str='primary', truncate_log: bool=False) -> subprocess.Popen:
        cmd = self._build_server_cmd(safer=safer)
        env = self._server_env(safer=safer)
        log_mode = 'w' if truncate_log else 'a'
        self.log_file = open('/kaggle/working/vllm_server.log', log_mode, encoding='utf-8')
        self.log_file.write(f'\n\n===== vLLM launch attempt: {attempt_name} | safer={safer} =====\n')
        self.log_file.flush()
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True, env=env)

    def _wait_for_server(self) -> None:
        start_time = time.time()
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
            if return_code is not None:
                if self.log_file is not None:
                    self.log_file.flush()
                logs = self._tail_logs()
                raise RuntimeError(f'vLLM server died with code {return_code}. Recent logs:\n{logs}')
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                return
            except Exception:
                time.sleep(1)
        if self.log_file is not None:
            self.log_file.flush()
        logs = self._tail_logs()
        raise TimeoutError(f'vLLM server did not become ready in time. Recent logs:\n{logs}')

    def _launch_server_with_retries(self) -> None:
        launch_plans = [{'attempt_name': 'primary', 'safer': False}, {'attempt_name': 'fallback', 'safer': True}]
        last_error = None
        for launch_index, plan in enumerate(launch_plans):
            self._cleanup_server()
            try:
                self.server_process = self._start_server(safer=plan['safer'], attempt_name=plan['attempt_name'], truncate_log=launch_index == 0)
                self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)
                self._wait_for_server()
                return
            except Exception as exc:
                last_error = exc
                self._cleanup_server()
                time.sleep(3)
        raise RuntimeError(f'All vLLM launch attempts failed. Last error:\n{last_error}')

    def _initialize_kernels(self) -> None:
        start_time = time.time()
        self.sandbox_pool = queue.Queue()
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(AIMO3Sandbox, self.cfg.jupyter_timeout) for _ in range(self.cfg.workers)]
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
        elapsed = time.time() - start_time

    @staticmethod
    def _assistant_tool_message(response_message: Any) -> dict:
        return {'role': 'assistant', 'content': response_message.content or '', 'tool_calls': [tc.model_dump() for tc in response_message.tool_calls or []]}

    @staticmethod
    def _extract_tool_code(arguments: str) -> str:
        if not arguments:
            return ''
        try:
            data = json.loads(arguments)
            if isinstance(data, dict):
                return str(data.get('code', ''))
        except Exception:
            pass
        match = re.search('"code"\\s*:\\s*"(.*)"\\s*}', arguments, re.S)
        if match:
            raw = match.group(1)
            try:
                return bytes(raw, 'utf-8').decode('unicode_escape')
            except Exception:
                return raw
        return arguments

    def _is_valid_answer(self, value: Optional[int]) -> bool:
        return value is not None and self.cfg.min_answer <= value <= self.cfg.max_answer

    def _scan_for_answer(self, text: str) -> Optional[int]:
        if not text:
            return None
        boxed_matches = re.findall('\\\\boxed\\s*\\{\\s*([0-9][0-9,\\s]{0,15})\\s*\\}', text)
        if boxed_matches:
            raw_value = boxed_matches[-1].replace(',', '').replace(' ', '')
            if raw_value.isdigit():
                value = int(raw_value)
                return value if self._is_valid_answer(value) else None
        explicit_matches = re.findall('(?:final\\s+answer|answer)\\s*(?:is|=|:)\\s*([0-9][0-9,\\s]{0,15})\\b', text, flags=re.IGNORECASE)
        if explicit_matches:
            raw_value = explicit_matches[-1].replace(',', '').replace(' ', '')
            if raw_value.isdigit():
                value = int(raw_value)
                return value if self._is_valid_answer(value) else None
        stripped = text.strip()
        if re.fullmatch('[0-9]{1,5}', stripped):
            value = int(stripped)
            return value if self._is_valid_answer(value) else None
        return None

    def _extract_entropy_weight(self, choice: Any) -> float:
        try:
            logprobs = getattr(choice, 'logprobs', None)
            if logprobs is None:
                return 1.0
            content = getattr(logprobs, 'content', None)
            if not content:
                return 1.0
            total_entropy = 0.0
            token_count = 0
            for token_info in content:
                top = getattr(token_info, 'top_logprobs', None)
                if not top:
                    continue
                probs = []
                for item in top:
                    lp = getattr(item, 'logprob', None)
                    if lp is None:
                        continue
                    probs.append(math.exp(lp))
                if not probs:
                    continue
                z = sum(probs)
                if z <= 0:
                    continue
                probs = [p / z for p in probs]
                entropy = -sum((p * math.log2(p) for p in probs if p > 0))
                total_entropy += entropy
                token_count += 1
            if token_count == 0:
                return 1.0
            mean_entropy = total_entropy / token_count
            return 1.0 / max(mean_entropy, 1e-06)
        except Exception:
            return 1.0

    @staticmethod
    def _approx_text_tokens(text: Any) -> int:
        if text is None:
            return 0
        normalized = text if isinstance(text, str) else json.dumps(text, ensure_ascii=False)
        if not normalized:
            return 0
        return max(1, math.ceil(len(normalized) / 4))

    def _estimate_message_tokens(self, message: dict) -> int:
        total = 8
        content = message.get('content')
        if isinstance(content, str):
            total += self._approx_text_tokens(content)
        elif isinstance(content, list):
            for item in content:
                if isinstance(item, dict):
                    total += self._approx_text_tokens(item.get('text') or item)
                else:
                    total += self._approx_text_tokens(item)
        elif content is not None:
            total += self._approx_text_tokens(content)
        tool_calls = message.get('tool_calls') or []
        if tool_calls:
            total += self._approx_text_tokens(tool_calls)
        tool_call_id = message.get('tool_call_id')
        if tool_call_id:
            total += self._approx_text_tokens(tool_call_id)
        return total

    def _estimate_prompt_tokens(self, messages: list[dict], tool: AIMO3Tool | None) -> int:
        estimate = 32 + sum((self._estimate_message_tokens(message) for message in messages))
        if tool is not None:
            estimate += self._approx_text_tokens(tool.tools)
        return estimate

    def _next_prompt_estimate(self, response: Any, appended_messages: list[dict], previous_estimate: int, used_tools: bool) -> int:
        usage = getattr(response, 'usage', None)
        prompt_tokens = getattr(usage, 'prompt_tokens', None) if usage is not None else None
        completion_tokens = int(getattr(usage, 'completion_tokens', 0) or 0) if usage is not None else 0
        if prompt_tokens is None:
            base = previous_estimate + completion_tokens
        else:
            base = int(prompt_tokens) + completion_tokens
        base += sum((self._estimate_message_tokens(message) for message in appended_messages))
        base += self.cfg.tool_turn_buffer_tokens if used_tools else self.cfg.chat_turn_buffer_tokens
        return base

    def _compute_turn_max_tokens(self, plan: dict, prompt_tokens_estimate: int) -> Optional[int]:
        remaining = self.cfg.context_tokens - int(prompt_tokens_estimate) - self.cfg.context_buffer_tokens
        if remaining < self.cfg.min_generation_tokens:
            return None
        return min(plan['max_tokens'], remaining)

    def _chat_completion(self, messages: list[dict], plan: dict, attempt_seed: int, tool: AIMO3Tool | None, max_tokens: int):
        extra_body = {'chat_template_kwargs': {'enable_thinking': True}}
        kwargs = {'model': self.cfg.served_model_name, 'messages': messages, 'temperature': plan['temperature'], 'top_p': plan['top_p'], 'max_tokens': max_tokens, 'seed': attempt_seed, 'extra_body': extra_body}
        if tool is not None and plan['use_tools']:
            kwargs['tools'] = tool.tools
            kwargs['tool_choice'] = 'auto'
        if self.cfg.logprobs:
            kwargs['logprobs'] = True
            kwargs['top_logprobs'] = self.cfg.top_logprobs
        try:
            return self.client.chat.completions.create(**kwargs)
        except Exception as exc:
            message = str(exc).lower()
            if 'logprob' in message:
                kwargs.pop('logprobs', None)
                kwargs.pop('top_logprobs', None)
                return self.client.chat.completions.create(**kwargs)
            raise

    def _finalize_from_messages(self, messages: list[dict], attempt_seed: int) -> Optional[int]:
        try:
            final_messages = list(messages)
            final_messages.append({'role': 'user', 'content': self.cfg.finalization_prompt})
            response = self.client.chat.completions.create(model=self.cfg.served_model_name, messages=final_messages, temperature=0.0, top_p=1.0, max_tokens=self.cfg.finalization_max_tokens, seed=attempt_seed, extra_body={'chat_template_kwargs': {'enable_thinking': False}})
            text = response.choices[0].message.content or ''
            return self._scan_for_answer(text)
        except Exception:
            return None

    def _process_attempt(self, problem: str, plan: dict, attempt_index: int, stop_event: threading.Event, deadline: float) -> dict:
        if stop_event.is_set() or time.time() > deadline:
            return {'Attempt': attempt_index + 1, 'Plan': plan['name'], 'Answer': None, 'Python Calls': 0, 'Python Errors': 0, 'Completion Tokens': 0, 'Entropy Weight': 0.0, 'Score': 0.0}
        sandbox = None
        tool = None
        python_calls = 0
        python_errors = 0
        total_completion_tokens = 0
        entropy_weight = 0.0
        final_answer = None
        attempt_seed = int((self.cfg.seed + attempt_index + 1) ** 2)
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            tool = AIMO3Tool(local_jupyter_timeout=self.cfg.jupyter_timeout, sandbox=sandbox) if plan['use_tools'] else None
            supported_tool_names = set(tool.supported_tool_names) if tool is not None else {'python_exec', 'stateful_python_code_exec'}
            user_prompt = problem.strip() + '\n\n' + plan['prompt_suffix']
            messages = [{'role': 'system', 'content': self.cfg.system_prompt}, {'role': 'user', 'content': user_prompt}]
            next_prompt_tokens_estimate = self._estimate_prompt_tokens(messages, tool)
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
                turn_max_tokens = self._compute_turn_max_tokens(plan, next_prompt_tokens_estimate)
                if turn_max_tokens is None:
                    break
                response = self._chat_completion(messages, plan, attempt_seed, tool, max_tokens=turn_max_tokens)
                choice = response.choices[0]
                message = choice.message
                entropy_weight += self._extract_entropy_weight(choice)
                usage = getattr(response, 'usage', None)
                if usage is not None and getattr(usage, 'completion_tokens', None) is not None:
                    total_completion_tokens += int(usage.completion_tokens)
                tool_calls = message.tool_calls or []
                if tool_calls:
                    python_calls += len(tool_calls)
                    tool_messages = []
                    for tc in tool_calls:
                        if tc.function.name not in supported_tool_names:
                            output = f'[ERROR] Unsupported tool: {tc.function.name}'
                            python_errors += 1
                        else:
                            code = self._extract_tool_code(tc.function.arguments or '')
                            try:
                                output = tool.run(code) if tool is not None else '[ERROR] Tool requested in no-tool plan.'
                            except Exception as exc:
                                output = f'[ERROR] {exc}'
                            if output.startswith('[ERROR]') or 'Traceback' in output or 'Error:' in output:
                                python_errors += 1
                        tool_messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': output})
                    assistant_message = self._assistant_tool_message(message)
                    messages.append(assistant_message)
                    messages.extend(tool_messages)
                    next_prompt_tokens_estimate = self._next_prompt_estimate(response=response, appended_messages=[assistant_message, *tool_messages], previous_estimate=next_prompt_tokens_estimate, used_tools=True)
                    continue
                content = message.content or ''
                assistant_message = {'role': 'assistant', 'content': content}
                messages.append(assistant_message)
                final_answer = self._scan_for_answer(content)
                if self._is_valid_answer(final_answer):
                    break
                progress_message = {'role': 'user', 'content': self.cfg.progress_prompt}
                messages.append(progress_message)
                next_prompt_tokens_estimate = self._next_prompt_estimate(response=response, appended_messages=[assistant_message, progress_message], previous_estimate=next_prompt_tokens_estimate, used_tools=False)
            if not self._is_valid_answer(final_answer) and (not stop_event.is_set()) and (time.time() <= deadline):
                final_answer = self._finalize_from_messages(messages, attempt_seed)
        except Exception:
            python_errors += 1
        finally:
            if sandbox is not None:
                with contextlib.suppress(Exception):
                    sandbox.reset()
                self.sandbox_pool.put(sandbox)
        score = 0.0
        if final_answer is not None:
            score += 2.0
        score += entropy_weight
        score += 0.15 * python_calls
        score -= 0.35 * python_errors
        score += 1.0 / max(total_completion_tokens, 1)
        return {'Attempt': attempt_index + 1, 'Plan': plan['name'], 'Answer': final_answer, 'Python Calls': python_calls, 'Python Errors': python_errors, 'Completion Tokens': total_completion_tokens, 'Entropy Weight': entropy_weight, 'Score': score}

    def _select_answer(self, detailed_results: list[dict]) -> Optional[int]:
        answer_votes = Counter()
        answer_scores = defaultdict(float)
        for result in detailed_results:
            answer = result['Answer']
            if answer is None:
                continue
            answer_votes[answer] += 1
            answer_scores[answer] += result['Score']
        if not answer_votes:
            return None
        rows = []
        for answer, votes in answer_votes.items():
            rows.append({'Answer': answer, 'Votes': votes, 'Weighted Score': answer_scores[answer]})
        vote_df = pd.DataFrame(rows).sort_values(['Votes', 'Weighted Score'], ascending=False)
        vote_df['Weighted Score'] = vote_df['Weighted Score'].round(4)
        display(vote_df)
        best = vote_df.iloc[0]
        return int(best['Answer'])

    def solve_problem(self, problem: str) -> int:
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
        deadline = time.time() + budget
        detailed_results: list[dict] = []
        valid_answers: list[int] = []
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
        try:
            futures = []
            for attempt_index, plan in enumerate(self.cfg.attempt_plans):
                future = executor.submit(self._process_attempt, problem, plan, attempt_index, stop_event, deadline)
                futures.append(future)
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
                    if valid_answers:
                        counts = Counter(valid_answers).most_common(3)
                except Exception as exc:
                    continue
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)
        if detailed_results:
            df = pd.DataFrame(detailed_results).sort_values('Attempt')
            if 'Entropy Weight' in df.columns:
                df['Entropy Weight'] = df['Entropy Weight'].round(4)
            if 'Score' in df.columns:
                df['Score'] = df['Score'].round(4)
            display(df)
        selected_answer = self._select_answer(detailed_results)
        if selected_answer is None:
            selected_answer = 0
        return int(selected_answer)

    def close(self) -> None:
        if getattr(self, '_closed', False):
            return
        self._closed = True
        self._cleanup_server()

    def __del__(self):
        with contextlib.suppress(Exception):
            self._cleanup_server()

In [ ]:
solver = AIMO3Solver(CFG)


In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    row_id = id_.item(0)
    problem_text = question.item(0)

    final_answer = 0
    gc.disable()
    try:
        final_answer = solver.solve_problem(problem_text)
    finally:
        gc.enable()
        gc.collect()

    return pl.DataFrame({"id": row_id, "answer": int(final_answer)})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

def _pick_local_gateway_csv() -> str:
    env_path = os.getenv('AIMO_LOCAL_CSV')
    if env_path:
        return env_path
    candidate_paths = ['/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv']
    for candidate in candidate_paths:
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError('Could not find a local AIMO csv. Set AIMO_LOCAL_CSV to reference.csv or test.csv explicitly.')
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    local_csv = _pick_local_gateway_csv()
    inference_server.run_local_gateway((local_csv,))